In [6]:
import random
import numpy as np
from datetime import date, timedelta

def generate_oura_sql(username: str, days: int = 30, seed: int = 42) -> str:
    """Generate realistic synthetic Oura data and return a SQL insert script."""
    random.seed(seed)
    np.random.seed(seed)

    # Look up the user_id from Supabase
    res = supabase.table("users").select("id").eq("username", username).execute()
    if not res.data:
        raise ValueError(f"User '{username}' not found.")
    user_id = res.data[0]["id"]

    rows = []
    # Simulate a slowly drifting baseline so data looks natural over 30 days
    baseline_sleep      = np.random.uniform(72, 82)
    baseline_readiness  = np.random.uniform(68, 78)
    baseline_activity   = np.random.uniform(65, 80)
    baseline_hrv        = np.random.uniform(38, 55)
    baseline_sleep_hrs  = np.random.uniform(6.5, 7.8)

    for i in range(days - 1, -1, -1):
        entry_date = (date.today() - timedelta(days=i)).isoformat()

        # Add day-to-day variation with a slight random walk
        baseline_sleep     += np.random.uniform(-2, 2)
        baseline_readiness += np.random.uniform(-2, 2)
        baseline_activity  += np.random.uniform(-3, 3)
        baseline_hrv       += np.random.uniform(-1.5, 1.5)
        baseline_sleep_hrs += np.random.uniform(-0.2, 0.2)

        # Clamp to realistic ranges
        sleep_score      = int(np.clip(baseline_sleep      + np.random.normal(0, 4), 45, 99))
        readiness_score  = int(np.clip(baseline_readiness  + np.random.normal(0, 4), 40, 99))
        activity_score   = int(np.clip(baseline_activity   + np.random.normal(0, 6), 30, 99))
        hrv_avg          = round(np.clip(baseline_hrv      + np.random.normal(0, 3), 18, 90), 1)
        sleep_hours      = round(np.clip(baseline_sleep_hrs + np.random.normal(0, 0.3), 4.0, 10.0), 2)

        # Deep and REM sleep correlate loosely with sleep quality
        sleep_quality_factor = sleep_score / 100
        deep_sleep_min = int(np.clip(
            np.random.normal(70 * sleep_quality_factor, 12), 20, 130
        ))
        rem_sleep_min = int(np.clip(
            np.random.normal(90 * sleep_quality_factor, 15), 30, 160
        ))

        rows.append({
            "entry_date":       entry_date,
            "sleep_score":      sleep_score,
            "readiness_score":  readiness_score,
            "activity_score":   activity_score,
            "hrv_avg":          hrv_avg,
            "sleep_hours":      sleep_hours,
            "deep_sleep_min":   deep_sleep_min,
            "rem_sleep_min":    rem_sleep_min,
        })

    # Build SQL
    lines = [
        "-- Synthetic Oura data for testing",
        f"-- User: {username} ({user_id})",
        f"-- Generated: {date.today().isoformat()}",
        "",
        "INSERT INTO oura_daily (",
        "  user_id, entry_date, sleep_score, readiness_score, activity_score,",
        "  hrv_avg, sleep_hours, deep_sleep_min, rem_sleep_min",
        ") VALUES",
    ]

    value_rows = []
    for r in rows:
        value_rows.append(
            f"  ('{user_id}', '{r['entry_date']}', {r['sleep_score']}, "
            f"{r['readiness_score']}, {r['activity_score']}, "
            f"{r['hrv_avg']}, {r['sleep_hours']}, "
            f"{r['deep_sleep_min']}, {r['rem_sleep_min']})"
        )

    lines.append(",\n".join(value_rows))
    lines.append("ON CONFLICT (user_id, entry_date) DO UPDATE SET")
    lines.append("  sleep_score     = EXCLUDED.sleep_score,")
    lines.append("  readiness_score = EXCLUDED.readiness_score,")
    lines.append("  activity_score  = EXCLUDED.activity_score,")
    lines.append("  hrv_avg         = EXCLUDED.hrv_avg,")
    lines.append("  sleep_hours     = EXCLUDED.sleep_hours,")
    lines.append("  deep_sleep_min  = EXCLUDED.deep_sleep_min,")
    lines.append("  rem_sleep_min   = EXCLUDED.rem_sleep_min,")
    lines.append("  updated_at      = now();")

    return "\n".join(lines)


# Generate and print the SQL
sql = generate_oura_sql("test1", days=30)
print(sql)

# Optionally save to file
with open("seed_oura_data.sql", "w") as f:
    f.write(sql)
print("\n✓ Saved to seed_oura_data.sql")

-- Synthetic Oura data for testing
-- User: test1 (dc7c7701-d24c-4c8a-8a51-f463e426a6b6)
-- Generated: 2026-04-30

INSERT INTO oura_daily (
  user_id, entry_date, sleep_score, readiness_score, activity_score,
  hrv_avg, sleep_hours, deep_sleep_min, rem_sleep_min
) VALUES
  ('dc7c7701-d24c-4c8a-8a51-f463e426a6b6', '2026-04-01', 72, 77, 75, 47.1, 6.86, 27, 38),
  ('dc7c7701-d24c-4c8a-8a51-f463e426a6b6', '2026-04-02', 72, 71, 80, 55.8, 6.89, 51, 57),
  ('dc7c7701-d24c-4c8a-8a51-f463e426a6b6', '2026-04-03', 71, 71, 76, 55.0, 6.89, 37, 76),
  ('dc7c7701-d24c-4c8a-8a51-f463e426a6b6', '2026-04-04', 67, 73, 85, 45.5, 6.6, 56, 52),
  ('dc7c7701-d24c-4c8a-8a51-f463e426a6b6', '2026-04-05', 69, 74, 86, 50.2, 6.95, 44, 67),
  ('dc7c7701-d24c-4c8a-8a51-f463e426a6b6', '2026-04-06', 75, 73, 80, 43.8, 7.27, 64, 79),
  ('dc7c7701-d24c-4c8a-8a51-f463e426a6b6', '2026-04-07', 72, 69, 79, 41.0, 7.2, 54, 86),
  ('dc7c7701-d24c-4c8a-8a51-f463e426a6b6', '2026-04-08', 68, 71, 84, 46.8, 7.53, 64, 63),
  ('dc7c77

In [4]:
# Verify the oura_daily table has data
# Check raw Supabase response
# 3. Try fetching oura_daily WITHOUT the user_id filter or date filter
test = (
    supabase.table("oura_daily")
    .select("entry_date, sleep_score")
    .execute()
)
print(f"Total rows (no filter): {len(test.data)}") # Print first 2 rows

Total rows (no filter): 30


In [2]:
import pandas as pd
from supabase import create_client

# Initialize Supabase client
SUPABASE_URL = "https://bfxpqmkhwgdsyvzzxnjk.supabase.co"
SUPABASE_KEY = "sb_publishable_9q666dvsL09qnBjhxOvYvg_IvKrrctf"
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

def load_reflections_df(user_id: str, days: int = 30) -> pd.DataFrame:
    """Load reflections + Oura data joined on user_id and entry_date."""
    from datetime import date, timedelta

    cutoff = (date.today() - timedelta(days=days)).isoformat()

    # Pull reflections
    reflections_res = (
        supabase.table("reflections")
        .select("entry_date, content, mood, keywords")
        .eq("user_id", user_id)
        .gte("entry_date", cutoff)
        .order("entry_date", desc=True)
        .execute()
    )

    # Pull Oura data
    oura_res = (
        supabase.table("oura_daily")
        .select("entry_date, sleep_score, readiness_score, activity_score, hrv_avg, sleep_hours, deep_sleep_min, rem_sleep_min")
        .eq("user_id", user_id)
        .gte("entry_date", cutoff)
        .execute()
    )

    # Build DataFrames
    df_reflections = pd.DataFrame(reflections_res.data or [])
    df_oura        = pd.DataFrame(oura_res.data or [])

    if df_reflections.empty:
        print("No reflection entries found.")
        return pd.DataFrame()

    # Clean dtypes
    df_reflections["entry_date"] = pd.to_datetime(df_reflections["entry_date"])
    df_reflections["mood"]       = pd.to_numeric(df_reflections["mood"], errors="coerce")

    if not df_oura.empty:
        df_oura["entry_date"] = pd.to_datetime(df_oura["entry_date"])

    # Left join — keeps all reflection days even if no Oura data yet
    df = pd.merge(df_reflections, df_oura, on="entry_date", how="left")

    return df


# Usage
res = supabase.table("users").select("id").eq("username", "test1").execute()
USER_ID = res.data[0]["id"]

df = load_reflections_df(USER_ID, days=30)
df.head()

,entry_date,content,mood,keywords,sleep_score,readiness_score,activity_score,hrv_avg,sleep_hours,deep_sleep_min,rem_sleep_min
0,2026-04-30,"Today, I am feeling burnt out from work. I can...",7.5,"[any development, focus, today, security reaso...",70.0,70.0,82.0,41.2,7.26,45.0,75.0
1,2026-04-27,I had a great weekend vending with Meeshka. It...,10.0,"[my jewelry, creative projects, meeshka, creat...",69.0,68.0,85.0,47.5,6.95,49.0,81.0
2,2026-04-22,I took time off the past two days. I have been...,9.5,"[these efforts, the mirra app, time, jewelry, ...",75.0,69.0,63.0,42.1,7.51,48.0,62.0
3,2026-04-13,"yesterday, there was some confusion of plans w...",6.0,"[yesterday, liv, a great brunch, some confusio...",70.0,65.0,75.0,45.6,7.10,64.0,38.0
4,2026-04-11,"Today, I worked on Lehua fest and crafted with...",9.5,"[lehua, meeshka, creative projects, today]",73.0,61.0,81.0,49.3,7.48,40.0,86.0


In [15]:
from sklearn.feature_extraction.text import CountVectorizer

def extract_phrases(df: pd.DataFrame, top_n: int = 30, min_count: int = 2) -> pd.DataFrame:
    """Extract top 2-4 word phrases using CountVectorizer."""

    if df.empty or "content" not in df.columns:
        print("No content column found.")
        return pd.DataFrame()

    texts = df["content"].dropna().astype(str).tolist()
    texts = [t.strip() for t in texts if t.strip()]

    if not texts:
        print("No content found.")
        return pd.DataFrame()

    vectorizer = CountVectorizer(
        ngram_range=(2, 4),       # 2 to 4 word phrases
        stop_words="english",     # remove common words like "i", "the", "it"
        min_df=min_count,         # only keep phrases appearing in N+ entries
        max_features=500,         # cap total phrases considered
        token_pattern=r"[a-zA-Z]{2,}"  # letters only, min 2 chars per word
    )

    try:
        X = vectorizer.fit_transform(texts)
    except ValueError as e:
        print(f"Vectorizer error: {e}")
        return pd.DataFrame()

    # Sum counts across all documents
    phrase_counts = X.sum(axis=0).A1
    phrases       = vectorizer.get_feature_names_out()

    phrase_df = (
        pd.DataFrame({"phrase": phrases, "count": phrase_counts})
        .sort_values("count", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    # Add date context — which entries contain each top phrase
    top_phrases = phrase_df["phrase"].tolist()
    date_map    = {p: [] for p in top_phrases}

    for _, row in df.iterrows():
        text = str(row.get("content", "")).lower()
        date = str(row.get("entry_date", ""))[:10]
        for phrase in top_phrases:
            if phrase in text:
                date_map[phrase].append(date)

    phrase_df["dates"]      = phrase_df["phrase"].map(lambda p: ", ".join(sorted(set(date_map[p]))))
    phrase_df["first_seen"] = phrase_df["phrase"].map(lambda p: min(date_map[p]) if date_map[p] else "")
    phrase_df["last_seen"]  = phrase_df["phrase"].map(lambda p: max(date_map[p]) if date_map[p] else "")

    return phrase_df


# Usage
phrase_df = extract_phrases(df, top_n=30, min_count=1)
phrase_df

,phrase,count,dates,first_seen,last_seen
0,creative projects,4,"2026-04-11, 2026-04-22, 2026-04-27, 2026-04-30",2026-04-11,2026-04-30
1,lehua fest,3,"2026-04-09, 2026-04-11, 2026-04-22",2026-04-09,2026-04-22
2,feel better,2,"2026-04-08, 2026-04-13",2026-04-08,2026-04-13
3,mirra app,2,"2026-04-09, 2026-04-22",2026-04-09,2026-04-22
4,ala wai water,1,,,
5,ala wai water helps,1,,,
6,app lehua,1,,,
7,app lehua fest,1,,,
8,app lehua fest happy,1,,,
9,app stats,1,,,


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def extract_phrases(df: pd.DataFrame, top_n: int = 30, min_count: int = 1) -> pd.DataFrame:
    """Extract top 2-4 word phrases using CountVectorizer."""

    if df.empty or "content" not in df.columns:
        print("No content column found.")
        return pd.DataFrame()

    texts = df["content"].dropna().astype(str).tolist()
    texts = [t.strip() for t in texts if t.strip()]

    if not texts:
        print("No content found.")
        return pd.DataFrame()

    vectorizer = CountVectorizer(
        ngram_range=(2, 4),       # 2 to 4 word phrases
        stop_words="english",     # remove common words like "i", "the", "it"
        min_df=min_count,         # only keep phrases appearing in N+ entries
        max_features=500,         # cap total phrases considered
        token_pattern=r"[a-zA-Z]{2,}"  # letters only, min 2 chars per word
    )

    try:
        X = vectorizer.fit_transform(texts)
    except ValueError as e:
        print(f"Vectorizer error: {e}")
        return pd.DataFrame()

    # Sum counts across all documents
    phrase_counts = X.sum(axis=0).A1
    phrases       = vectorizer.get_feature_names_out()

    phrase_df = (
        pd.DataFrame({"phrase": phrases, "count": phrase_counts})
        .sort_values("count", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    # Add date context — which entries contain each top phrase
    top_phrases = phrase_df["phrase"].tolist()
    date_map    = {p: [] for p in top_phrases}

    for _, row in df.iterrows():
        text = str(row.get("content", "")).lower()
        date = str(row.get("entry_date", ""))[:10]
        for phrase in top_phrases:
            if phrase in text:
                date_map[phrase].append(date)

    phrase_df["dates"]      = phrase_df["phrase"].map(lambda p: ", ".join(sorted(set(date_map[p]))))
    phrase_df["first_seen"] = phrase_df["phrase"].map(lambda p: min(date_map[p]) if date_map[p] else "")
    phrase_df["last_seen"]  = phrase_df["phrase"].map(lambda p: max(date_map[p]) if date_map[p] else "")

    return phrase_df


# Usage
phrase_df = extract_phrases(df, top_n=50, min_count=1)
phrase_df

,phrase,count,first_seen,last_seen,dates
0,creative projects,4,2026-04-11,2026-04-30,"2026-04-11, 2026-04-22, 2026-04-27, 2026-04-30"
1,lehua fest,2,2026-04-09,2026-04-22,"2026-04-09, 2026-04-22"
2,any development,1,2026-04-30,2026-04-30,2026-04-30
3,security reasons,1,2026-04-30,2026-04-30,2026-04-30
4,creative flow,1,2026-04-27,2026-04-27,2026-04-27
5,great weekend,1,2026-04-27,2026-04-27,2026-04-27
6,some confusion,1,2026-04-13,2026-04-13,2026-04-13
7,better today,1,2026-04-13,2026-04-13,2026-04-13
8,great brunch,1,2026-04-13,2026-04-13,2026-04-13
9,mirra app,1,2026-04-09,2026-04-09,2026-04-09


In [14]:
len(phrase_df)

35

In [8]:
# 1. Check how many entries have actual content
print("Total rows:", len(df))
print("Rows with content:", df["content"].notna().sum())
print("Non-empty content:", (df["content"].str.strip() != "").sum())
print()

# 2. Preview the actual content
print(df["content"].dropna().head(5).tolist())

Total rows: 13
Rows with content: 13
Non-empty content: 13

['Today, I am feeling burnt out from work. I cannot do any development because of security reasons. I am shifting focus to creative projects. ', 'I had a great weekend vending with Meeshka. It was refreshing to invest in creative projects like selling my jewelry. I also got to fire spin both saturday and sunday; A lot of creative flow overall. ', 'I took time off the past two days. I have been focusing on creative projects such as making jewelry, the Mirra app, and Lehua Fest. I am much more happy to focus on these efforts. ', 'yesterday, there was some confusion of plans which was frustrating. I had a great brunch with Liv though. I feel better today. ', "Today, I worked on Lehua fest and crafted with Meeshka. I'm really happy to work on creative projects. "]


In [9]:
# 3. Check what raw noun chunks spaCy is finding before any filters
sample_text = df["content"].dropna().iloc[0]
print("Sample text:", sample_text)
print()

doc = nlp(str(sample_text))
print("All noun chunks found:")
for chunk in doc.noun_chunks:
    words = chunk.text.lower().strip().split()
    print(f"  '{chunk.text}' → {len(words)} words")
    

Sample text: Today, I am feeling burnt out from work. I cannot do any development because of security reasons. I am shifting focus to creative projects. 

All noun chunks found:
  'I' → 1 words
  'work' → 1 words
  'I' → 1 words
  'any development' → 2 words
  'security reasons' → 2 words
  'I' → 1 words
  'focus' → 1 words
  'creative projects' → 2 words


In [10]:
# 4. Relax ALL filters and see what comes out
phrase_freq = {}
for _, row in df.iterrows():
    text = row.get("content", "")
    if not text or not str(text).strip():
        continue
    doc = nlp(str(text))
    for chunk in doc.noun_chunks:
        phrase = chunk.text.lower().strip()
        phrase_freq[phrase] = phrase_freq.get(phrase, 0) + 1

print(f"Total phrases with NO filters: {len(phrase_freq)}")
print(sorted(phrase_freq.items(), key=lambda x: x[1], reverse=True)[:20])


Total phrases with NO filters: 79
[('i', 19), ('it', 7), ('creative projects', 4), ('which', 4), ('meeshka', 2), ('lehua fest', 2), ('mirra', 2), ('work', 1), ('any development', 1), ('security reasons', 1), ('focus', 1), ('a great weekend', 1), ('my jewelry', 1), ('spin', 1), ('a lot', 1), ('creative flow', 1), ('time', 1), ('the past two days', 1), ('jewelry', 1), ('the mirra app', 1)]
